##### Library imports

In [ ]:
import re, os, sys
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
import numpy as np 
from skimage import measure, filters
import statsmodels.api as sm
import matplotlib.pyplot as plt
import json

##### Function definitions

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def moore_contour(blob):
    """
    Implements the Moore neighborhood contour tracing algorithm to detect 
    and return the boundary pixels of a binary blob in an image.

    Parameters
    ----------
    blob : numpy.ndarray
        A 2D binary array representing the input image mask, where the 
        foreground (blob) is represented by 1s and background by 0s.

    Returns
    -------
    contour : numpy.ndarray
        A 2D binary array of the same shape as the input `blob`, where 
        only the boundary pixels of the detected contour are set to 1.

    Raises
    ------
    ValueError
        If the input `blob` is a single pixel (shape `(1, 1)`).
    """

    if blob.shape == (1,1):
        raise ValueError('Single pixel in segmentation, inspect processing up to here.')
        
    blob_padded = np.pad(blob, (1,1), mode = 'constant', constant_values = (0,0))
    boundary_pts = []
    [row,col] = blob_padded.nonzero()
    start_row, start_col = min(row[col==min(col)]), min(col) #start on the top left of the image
    boundary_pts.append((start_row,start_col)) #add first boundary to the list

    #without loss of generality, assume that we started from the point on the left to arrive here
    prev_row, prev_col = start_row, start_col - 1 

    #this shows the initial offset of (0,-1) and records the "backwards" step taken from previous point to current point 
    d_row, d_col = prev_row - start_row, prev_col - start_col 

    #define the 8 points in the Moore neighborhood
    moore_neighborhood = [(0,-1),(-1,-1),(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1)]

    count = 1
    
    #current boundary point we are on 
    b_row, b_col = start_row, start_col 

    while count < 2: #algorithm terminates when the same point is visited twice
        for step in range(len(moore_neighborhood)):
            if (d_row,d_col) == moore_neighborhood[step]:
                step_num = step
                break
                
        #This loop executes until a foreground pixel is found
        while True:
            #back-track from current foreground pixel and advance one pixel in the clock-wise direction
            cur_row, cur_col = b_row + moore_neighborhood[step_num][0], b_col + moore_neighborhood[step_num][1]
            if blob_padded[cur_row, cur_col] == 1:           
                if ((cur_row, cur_col) in boundary_pts) == True:
                    count += 1
                    if count == 2: 
                        break #terminates the algorithm when the same pixel is visited twice
                boundary_pts.append((cur_row,cur_col))
                #instead of going all the way again, record the back-step here
                d_row, d_col = prev_row - cur_row, prev_col - cur_col
                b_row, b_col = cur_row, cur_col
                break #break loop when foreground pixel is found
            prev_row, prev_col = cur_row, cur_col
            step_num += 1
            #This reset is necessary to do the moore-neighborhood search
            if step_num==8:
                step_num = 0
                
    #drawing the detected contour
    moore_boundary_pts = [(x-1,y-1) for (x,y) in boundary_pts] #because  we padded zeros earlier, subtract 1 from both indices
    contour = np.zeros(blob.shape)
    for pts in moore_boundary_pts:
        contour[pts[0],pts[1]] = 1
    return contour

In [ ]:
def rotate_image(image, dimension = 3):
    """Rotates NIfTI coronal images so they are upright for visualization. Simple insights tool kit (SITK) utilizes the LPS coordinate system, 
       meaning image voxel coordinates are assumed to increase in the Right --> Left, Anterior --> Posterior, and Inferior --> Superior directions. 
       However, NIfTI images use the RAS coordinate system, which makes coronal slices (vertical cross-sections) appear inverted when read with SITK. 
       This function resamples the image such that it is visualized in upright direction, enabling interpretable visualization and processing. 
    
    Parameters
    ----------
    image : sitk.Image
        The original NIfTI image (Note that the direction cosine should be [-1,0,0,0,-1,0,0,0,1], although this is a coronal image, it is 
        assumed to be resampled and cropped such that the direction cosine corresponds to an axial volume)
    dimension : int, optional
        Dimensionality of the original and rotation-corrected image. Default is 3.
    
    Returns
    -------
    rotated_image : sitk.Image
        Rotation-corrected image, with a direction cosine of [1,0,0,0,1,0,0,0,1]
    """

    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    s = image.GetSpacing()[2]

    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,-1.0]])

   
    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def num_connected_comps_cv(clean_ventricles, suppress_blob_area_ratio = 0.3, disconnectedLV_height_diff_tol = 5, 
                          disconnectedLV_area_diff_tol = 0.05, disconnectedLV_x_diff_tol = 20): 
    """
    Processes the Left Ventricle (LV) segmentation to remove extra ventricular tissue. 
    
    The crude segmentation of the left and right LVs may appear connected when they 
    are enlarged (e.g., NPH and AD) or may appear disconnected (mostly in HC and PTE).
    
    Parameters
    ----------
    clean_ventricles : numpy.ndarray
        A 2D binary or labeled array representing the initial LV segmentation.
    suppress_blob_area_ratio : float, optional
        Threshold ratio to suppress blobs occupying less than this fraction of 
        the maximum blob area. Default is 0.3 (30%).
    disconnectedLV_height_diff_tol : int or float, optional
        Maximum allowed difference in the vertical span (height) between two blobs 
        to be considered valid disconnected LVs. Default is 5.
    disconnectedLV_area_diff_tol : float, optional
        Minimum required area ratio of the smaller blob relative to the second blob 
        for the two blobs to be considered valid disconnected LVs. Default is 0.05 (5%).
    disconnectedLV_x_diff_tol : int or float, optional
        Maximum allowed horizontal distance from the image center for valid disconnected LV blobs. 
        Default is 20.

    Returns
    -------
    disconnected_case : bool
        True if the algorithm determines the segmentation represents disconnected 
        LVs, False otherwise.
    processed_blobs : numpy.ndarray
        A 2D binary array of the cleaned and processed ventricle segmentation.

    Raises
    ------
    ValueError
        If no segmented area remains after initial suppression, or if the algorithm 
        fails to suppress non-ventricular masses.
    """
    blobs_labels = measure.label(clean_ventricles, background=0)

    # Grab counts of all distinct blobs (except the background)
    blob_sums_unprocessed = np.bincount(blobs_labels.flatten())[1:]

    # For cases with more than 2 blobs excluding the background
    if len(blob_sums_unprocessed) > 2:
        # Suppress blobs which occupy less than the specified area ratio of the largest blob.
        max_blob_area = np.max(blob_sums_unprocessed)
        suppress_labels = [ind + 1 for ind in np.where(blob_sums_unprocessed / max_blob_area < suppress_blob_area_ratio)[0]]

        # Suppress labels for non-ventricular tissue
        for sl in suppress_labels:
            blobs_labels[blobs_labels == sl] = 0
            
    # Make sure there is at least 1 non-background connected component
    if len(np.unique(blobs_labels)) <= 1:
        raise ValueError("No segmented area remains, check processing up to here.")

    processed_blobs = blobs_labels

    # If there are more than 3 non-background disconnected components, keep the largest 3
    if len(np.unique(blobs_labels)) > 3: 
        top_3_args = np.argsort(np.bincount(blobs_labels.flatten())[1:])[::-1][:3]
        
        processed_blobs_1 = np.where(blobs_labels == top_3_args[0] + 1, blobs_labels, 0)
        pr1, _ = processed_blobs_1.nonzero()
        
        processed_blobs_2 = np.where(blobs_labels == top_3_args[1] + 1, blobs_labels, 0)
        pr2, _ = processed_blobs_2.nonzero()
        
        processed_blobs_3 = np.where(blobs_labels == top_3_args[2] + 1, blobs_labels, 0)
        pr3, _ = processed_blobs_3.nonzero()

        # Compare the y-direction medians to exclude the 3V (which sits lower)
        closest_inds = np.argmin([
            np.abs(np.median(pr1) - np.median(pr2)), 
            np.abs(np.median(pr1) - np.median(pr3)),
            np.abs(np.median(pr2) - np.median(pr3))
        ])
        
        if closest_inds == 0:
            processed_blobs = processed_blobs_1 + processed_blobs_2
        elif closest_inds == 1:
            processed_blobs = processed_blobs_1 + processed_blobs_3
        else:
            processed_blobs = processed_blobs_2 + processed_blobs_3 

    # There should ideally be no non-ventricular tissue here
    if len(np.unique(processed_blobs)) > 3:
        raise ValueError("Failed to suppress non-ventricular mass. Check segmentation up to here.")
        
    # Start by assuming a connected LV and an irrelevant non-background component
    disconnected_case = False

    # Make the blob labels contiguous using standard NumPy mapping
    rem_labels = np.unique(processed_blobs)
    mapped_blobs = np.zeros_like(processed_blobs)
    for new_val, old_val in enumerate(rem_labels):
        mapped_blobs[processed_blobs == old_val] = new_val
    processed_blobs = mapped_blobs

    # Check for any non-ventricular tissue based on blob height and laterality
    _, pc = processed_blobs.nonzero()
    im_cen = (np.max(pc) + np.min(pc)) // 2

    # If there are exactly 2 non-background blobs
    if len(np.unique(processed_blobs)) == 3: 
        counts = np.bincount(processed_blobs.flatten())
        
        smallest_blob_arg = np.argsort(counts)[0]
        smallest_blob = np.where(processed_blobs == smallest_blob_arg, 1, 0)
        sb_r, sb_c = smallest_blob.nonzero()
        smallest_blob_height = (np.min(sb_r) + np.max(sb_r)) // 2
        smallest_blob_cen_x = (np.min(sb_c) + np.max(sb_c)) // 2

        mid_blob_arg = np.argsort(counts)[1]
        mid_blob = np.where(processed_blobs == mid_blob_arg, 1, 0)
        mb_r, mb_c = mid_blob.nonzero()
        mid_blob_height = (np.min(mb_r) + np.max(mb_r)) // 2
        mid_blob_cen_x = (np.min(mb_c) + np.max(mb_c)) // 2

        # CONDITION 1: Difference in vertical span of the two blobs
        nv_mass = np.abs(smallest_blob_height - mid_blob_height) > disconnectedLV_height_diff_tol 

        # CONDITION 2: Smallest blob area ratio check
        area_check = np.sum(smallest_blob) < (disconnectedLV_area_diff_tol * np.sum(mid_blob))

        # CONDITION 1 AND CONDITION 2 should not hold, and both blobs should be centered: Declare disconnected LVs
        centered_small = np.abs(smallest_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol
        centered_mid = np.abs(mid_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol
        
        if centered_small and centered_mid and not nv_mass and not area_check:
            disconnected_case = True
        else: 
            # Suppress non-ventricular tissue
            processed_blobs = np.where(processed_blobs == smallest_blob_arg, 0, processed_blobs)

    # Binarize the processed image
    processed_blobs = np.where(processed_blobs > 0, 1, 0) 

    return disconnected_case, processed_blobs 

In [ ]:
def trace_left_contour(row, col, bot_con_row, ven_contour):

    """
    Traces the left contour of a ventricle segmentation moving in a primarily 
    South-East direction until a stopping row or a flat area limit is reached.
    
    Parameters
    ----------
    row : int
        The starting row index for the trace.
    col : int
        The starting column index for the trace.
    bot_con_row : int
        The target bottom row index (stopping point) for the trace. If the 
        starting `row` is greater than this, it is clamped to `bot_con_row`.
    ven_contour : numpy.ndarray
        A 2D binary array representing the contour of the ventricle.

    Returns
    -------
    left_pts : list of tuple of int
        A list of (x, y) coordinates representing the full trajectory path taken 
        by the algorithm.
    left_con_pts : list of tuple of int
        A list of (x, y) coordinates representing only the path points that 
        successfully landed on the active `ven_contour` mask.
    """
    left_pts = [] 
    left_con_pts = [] 
    flat_area_count = 0 

    height, width = ven_contour.shape

    #adjust starting row to match the centroid height
    if row > bot_con_row: 
        row = bot_con_row

    #trajectory is assumed to be in the SE/S/E direction 
    #this is essentially moving from the lateral and superior region to medial and inferior regions until we hit the stopping point
    #near the center. For suggested N/NE/NW/W/SW direction steps the algorithm is defaulted to an E step.

    while row <= bot_con_row and flat_area_count <= 10:   
        
        if len(left_pts) > 1:
            if left_pts[-1][1] == left_pts[-2][1]:
                flat_area_count += 1
            else: 
                flat_area_count = 0

        # Boundary flags to ensure we never check pixels outside the image
        can_move_south = (row + 1 < height)
        can_move_east = (col + 1 < width)

        # 1. South-East step
        if can_move_south and can_move_east and (ven_contour[row + 1, col + 1] > 0):
            left_pts.append((col + 1, row + 1))
            left_con_pts.append((col + 1, row + 1))
            row, col = row + 1, col + 1
            
        # 2. East step
        elif can_move_east and (ven_contour[row, col + 1] > 0):
            left_pts.append((col + 1, row))
            left_con_pts.append((col + 1, row))
            row, col = row, col + 1                        
            
        # 3. South step
        elif can_move_south and (ven_contour[row + 1, col] > 0): 
            left_pts.append((col, row + 1))
            left_con_pts.append((col, row + 1))
            row, col = row + 1, col                  
            
        # 4. Default East step 
        elif can_move_east: 
            left_pts.append((col + 1, row))
            row, col = row, col + 1

        # 5. absolute dead-end (e.g., bottom-right corner of the image)
        else:
            print(f"Warning: Contour tracing hit absolute image boundary at ({row}, {col}).")
            break

    return left_pts, left_con_pts

In [ ]:
def trace_right_contour(row, col, bot_con_row, ven_contour):    
    """
    Traces the right contour of a ventricle segmentation moving in a primarily 
    South-West direction until a stopping row or a flat area limit is reached.
    
    Parameters
    ----------
    row : int
        The starting row index for the trace.
    col : int
        The starting column index for the trace.
    bot_con_row : int
        The target bottom row index (stopping point) for the trace. If the 
        starting `row` is greater than this, it is clamped to `bot_con_row`.
    ven_contour : numpy.ndarray
        A 2D binary array representing the contour of the ventricle.

    Returns
    -------
    right_pts : list of tuple of int
        A list of (x, y) coordinates representing the full trajectory path taken 
        by the algorithm.
    right_con_pts : list of tuple of int
        A list of (x, y) coordinates representing only the path points that 
        successfully landed on the active `ven_contour` mask.
    """
    right_pts = []
    right_con_pts = []
    flat_area_count = 0
    
    height, width = ven_contour.shape

    #adjust starting row to match the centroid height
    if row > bot_con_row:
        row = bot_con_row

    #trajectory is assumed to be in the SW/S/W direction 
    #this is essentially moving from the lateral and superior region to medial and inferior regions until we hit the stopping point
    #near the center. For suggested N/NE/NW/E/SE direction steps the algorithm is defaulted to a W step. 
    
    while row <= bot_con_row and flat_area_count <= 10:  
        
        if len(right_pts) > 1:
            if right_pts[-1][1] == right_pts[-2][1]:
                flat_area_count += 1
            else: 
                flat_area_count = 0

        # Boundary flags to ensure we never check pixels outside the image
        can_move_south = (row + 1 < height)
        can_move_west = (col - 1 >= 0)
        
        if can_move_south and can_move_west and (ven_contour[row + 1,col - 1] > 0):#south-west
            right_pts.append((col - 1,row + 1))
            right_con_pts.append((col - 1,row + 1))
            row, col = row + 1, col - 1 
            
        elif can_move_west and (ven_contour[row, col - 1] > 0):#west
            right_pts.append((col - 1,row))
            right_con_pts.append((col - 1,row))
            row, col = row, col - 1   
            
        elif can_move_south and (ven_contour[row + 1,col] > 0): #south
            right_pts.append((col,row + 1))
            right_con_pts.append((col,row + 1))
            row, col = row + 1, col        

        # 4. Default West step 
        elif can_move_west: 
            right_pts.append((col - 1, row))
            row, col = row, col - 1
            
        # 5. absolute dead-end (e.g., bottom-right corner of the image)
        else:
            print(f"Warning: Contour tracing hit absolute image boundary at ({row}, {col}).")
            break
        
    return right_pts, right_con_pts

In [ ]:
def fit_lines(left_con_cand_pts,right_con_cand_pts):
    """
    Calculates the interior angle between Ordinary Least Squares (OLS) fit lines 
    traced on the medial walls of the left and right ventricles.
    
    Parameters
    ----------
    left_con_cand_pts : list of tuple of int or float
        A list of (x, y) coordinates representing the candidate contour points 
        for the left ventricle.
    right_con_cand_pts : list of tuple of int or float
        A list of (x, y) coordinates representing the candidate contour points 
        for the right ventricle.

    Returns
    -------
    angle : float
        The calculated interior angle (in degrees) between the two fitted lines.
    """
    
    x_l = [x for (x,y) in left_con_cand_pts]
    y_l = [y for (x,y) in left_con_cand_pts]
    x_l = sm.add_constant(x_l,has_constant='add')
    result_l = sm.OLS(y_l, x_l).fit()
    left_ven_slope = result_l.params[1]
    
    x_r = [x for (x,y) in right_con_cand_pts]
    y_r = [y for (x,y) in right_con_cand_pts]
    x_r = sm.add_constant(x_r,has_constant='add')
    result_r = sm.OLS(y_r, x_r).fit()
    right_ven_slope = result_r.params[1]
        
    return 180 - abs(np.degrees(np.arctan(right_ven_slope))) - abs(np.degrees(np.arctan(left_ven_slope)))

In [ ]:
def trace_ca_contour(lat_ventricles, coronal_plane, disconnected_case):

    """
    Traces the roof of the segmented lateral ventricles and calculates the 
    callosal angle by fitting lines to the medial walls.
    
    Parameters
    ----------
    lat_ventricles : numpy.ndarray
        A 2D binary array representing the segmented lateral ventricles.
    coronal_plane : numpy.ndarray
        A 2D array representing the original coronal anatomical image, used 
        for visualization.
    disconnected_case : bool
        Flag indicating whether the ventricles are disconnected (True) or 
        connected as a single blob (False).

    Returns
    -------
    ca_angle : float
        The calculated interior corpus callosum (CA) angle in degrees.
    """
    #centroid of the segmented image to help with stopping conditions
    M = cv.moments(np.uint8(lat_ventricles))
    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])

    #In case the LV is segmented as a connected blob, process it as it is
    if not(disconnected_case):
        ven_contour = moore_contour(lat_ventricles)
 
        [vc_r, vc_c] = ven_contour.nonzero()

        ##### Left LV Processing
        l_stop_pt = int((min(vc_c)+max(vc_c))//2)-2 #stop tracing the left LVs contour at this point

        #grab the left LV
        [lvc_r, lvc_c] = ven_contour[:,:l_stop_pt].nonzero()

        #If the left LV is not captured, move the end point to the image center
        if len(lvc_r) == 0:
            l_stop_pt = int((min(vc_c)+max(vc_c))//2)
            [lvc_r, lvc_c] = ven_contour[:,:l_stop_pt].nonzero()

        #Grab the top-most point of the left LV and the corresponding left-most point on that row. 
        lv_tp_y = min(lvc_r)
        lv_tp_x = min(lvc_c[lvc_r==lv_tp_y])

        #start contour tracing
        row, col = lv_tp_y, lv_tp_x
        bot_con_row = cY
        left_pts, left_con_pts = trace_left_contour(row, col, bot_con_row, ven_contour)

        ##### Right LV Processing
        r_start_pt = int((min(vc_c)+max(vc_c))//2)+2

        #grab the right LV
        [rvc_r, rvc_c] = ven_contour[:,r_start_pt:].nonzero()

        #If the right LV is not captured, move the starting point to the image center
        if len(rvc_r) == 0:
            r_start_pt = int((min(vc_c)+max(vc_c))//2)
            [rvc_r, rvc_c] = ven_contour[:,r_start_pt:].nonzero()

        #Grab the top-most point of the right LV and the corresponding right-most point on that row. 
        rv_tp_y = min(rvc_r)
        rv_tp_x = max(rvc_c[rvc_r==rv_tp_y]) + r_start_pt

        #start contour tracing
        row, col = rv_tp_y, rv_tp_x

        right_pts, right_con_pts = trace_right_contour(row, col, bot_con_row, ven_contour)
         

    else:
        #Get the border points of the LVs in the disconnected case, using the Moore Contour method. Since the algorithm works only on one blob, 
        #this just splits the image into the left and right LVs and applies Moore Contour tracing on them separately, combines the border into a 
        #full image

        [lat_r,lat_c] = lat_ventricles.nonzero()

        left_ven_full = lat_ventricles.copy()
        left_ven_full[:,(min(lat_c)+max(lat_c))//2:] = 0
        left_ven_full = getLargestCC(measure.label(left_ven_full,background = 0)) 

        right_ven_full = lat_ventricles.copy()
        right_ven_full[:,:(min(lat_c)+max(lat_c))//2] = 0
        right_ven_full = getLargestCC(measure.label(right_ven_full,background = 0)) 

        left_ven_con = moore_contour(left_ven_full)
        right_ven_con = moore_contour(right_ven_full)
        ven_contour = left_ven_con + right_ven_con

        ##### Left LV Processing
        [lv_r,lv_c] = left_ven_con.nonzero()
        lv_tp_y = min(lv_r)
        lv_tp_x = min(lv_c[lv_r==lv_tp_y]) + 2
        
        row, col = lv_tp_y, lv_tp_x
        bot_con_row = cY

        left_pts, left_con_pts = trace_left_contour(row, col, bot_con_row, ven_contour)

        ##### Right LV Processing
        [rv_r,rv_c] = right_ven_con.nonzero()
        rv_tp_y = min(rv_r)
        rv_tp_x = max(rv_c[rv_r==rv_tp_y]) - 2

        #start contour tracing
        row, col =  rv_tp_y,rv_tp_x        
        right_pts, right_con_pts = trace_right_contour(row, col, bot_con_row, ven_contour)


    ## Clean up left LV contour points by retaining only the ones placed on the medial wall's midsection
    left_min_row, left_max_row = left_con_pts[0][1],left_con_pts[-1][1]
    left_y_range = left_max_row - left_min_row
    left_low_thresh, left_high_thresh = left_max_row - 0.2*(left_y_range), left_min_row + 0.2*(left_y_range)            
    left_con_cand_pts = [(x,y) for x,y in left_con_pts if (y >= left_high_thresh) & (y <= left_low_thresh)]
    if len(left_con_cand_pts) < 5:
        try:
            left_con_cand_pts = left_pts[:np.where(np.diff([y for x,y in left_pts])>0)[0][-1]]
        except IndexError:
            left_con_cand_pts = left_pts

    ## Clean up right LV contour points by retaining only the ones placed on the medial wall's midsection
    right_min_row, right_max_row = right_con_pts[0][1],right_con_pts[-1][1]
    right_y_range = right_max_row - right_min_row
    right_low_thresh, right_high_thresh = right_max_row - 0.2*(right_y_range), right_min_row + 0.2*(right_y_range)            
    right_con_cand_pts = [(x,y) for x,y in right_con_pts if (y >= right_high_thresh) & (y <= right_low_thresh)]
    if len(right_con_cand_pts) < 5:
        try:
            right_con_cand_pts = right_pts[:np.where(np.diff([y for x,y in right_pts])>0)[0][-1]]
        except IndexError:
            right_con_cand_pts = right_pts

    ## Fit lines to the traced contour points and calculate angle between them
    ca_angle = fit_lines(left_con_cand_pts,right_con_cand_pts)
        
    fig, ax = plt.subplots(figsize = (12,12))
    ax.imshow(coronal_plane,cmap = 'gray')
    [ax.scatter(x,y,marker = 'x',color =   "w") for (x,y) in left_con_cand_pts]
    [ax.scatter(x,y,marker = 'x',color =   "w") for (x,y) in right_con_cand_pts]
    plt.show()
        
    return ca_angle

In [ ]:
def process_calculate_ca(cor_img_path, pc, processing_parameters, visualize_intermediate = False):

    """
    Orchestrates the image processing pipeline to segment the lateral ventricles 
    from a coronal slice at the Posterior Commissure (PC) level and compute the 
    Callosal Angle (CA).

    Parameters
    ----------
    cor_img_path : str
        File path to the input coronal medical image (e.g., NIfTI or DICOM).
    pc : tuple or list of float
        The 3D physical coordinates of the Posterior Commissure (PC) used to 
        locate the correct coronal slice.
    processing_parameters : dict
        A dictionary containing the necessary processing parameters for segmentation, ROI placement, and morphological operations.
    visualize_intermediate : bool, optional
        If True, displays sequential matplotlib plots of the segmentation 
        steps for debugging purposes. Default is False.

    Returns
    -------
    ca_angle : float
        The calculated interior corpus callosum (CA) angle in degrees. Returns 
        `np.nan` if the extraction or contour tracing pipeline fails.
    """

    #unpack processing parameters
    gauss_blur_kernel_size = processing_parameters['gauss_blur_kernel_size']
    adaptive_thresh_size = processing_parameters['adaptive_thresh_size']
    adaptive_thresh_C = processing_parameters['adaptive_thresh_C']
    window_center = processing_parameters['window_center']
    window_width = processing_parameters['window_width']
    img_dim = processing_parameters['img_dim']
    above_pc_mask_threshold = processing_parameters['above_pc_mask_threshold']
    middle_roi_xlims = processing_parameters['middle_roi_xlims']
    middle_roi_upper_ylim = processing_parameters['middle_roi_upper_ylim']
    smoothing_factor = processing_parameters['smoothing_factor']
    dec_smooth_factor_max_thresh = processing_parameters['dec_smooth_factor_max_thresh']
    initial_smooth_ven_fullbrain_ratio = processing_parameters['initial_smooth_ven_fullbrain_ratio']
    smoothed_ven_fullbrain_ratio = processing_parameters['smoothed_ven_fullbrain_ratio']
    median_blur_kernel_size = processing_parameters['median_blur_kernel_size']
    
    cor_img = sitk.ReadImage(cor_img_path)
    cor_img_brain = window_stack_sitk(rotate_image(cor_img,img_dim), window_center, window_width)
            
    cor_img_brain_dat = sitk.GetArrayFromImage(cor_img_brain)
    cor_pc_index = cor_img_brain.TransformPhysicalPointToContinuousIndex(pc)
    pl  = int(np.round(cor_pc_index[1])) #getting the y coordinate 
   
    #picking the standard plane at the PC level for measurement   
    coronal_plane = cor_img_brain_dat[:,pl,:] 
    
    #denoising
    coronal_gauss_blurred = cv.GaussianBlur(np.uint8(coronal_plane),
                                            (gauss_blur_kernel_size,gauss_blur_kernel_size), 
                                            cv.BORDER_DEFAULT) 
    
    #global intensity threshold determination for CSF vs brain
    otsu_thresh = filters.threshold_otsu(coronal_gauss_blurred) 
    coronal_bin = coronal_gauss_blurred > otsu_thresh    
    
    #flood filling to determine full brain mask         
    im_th = coronal_bin.copy()
    h, w = im_th.shape[:2]
    mask = np.zeros((h+2, w+2), np.uint8)
    im_floodfill = im_th.copy()        
    res = cv.floodFill(np.uint8(im_floodfill), mask, (0,0), 255) # Floodfill from point (0, 0)
    full_brain_mask = (1-res[2]).copy()
    full_brain_mask = full_brain_mask[1:h+1,1:w+1]
    
        
    #determine adaptive local thresholded image from the gauss blurred image
    coronal_adapt_thresh = cv.adaptiveThreshold(coronal_gauss_blurred, 1, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv.THRESH_BINARY, adaptive_thresh_size, adaptive_thresh_C)
    coronal_adapt_thresh_bg_supressed = np.where((1-full_brain_mask)>0,0,coronal_adapt_thresh)
    coronal_brain_mask = np.where(coronal_adapt_thresh_bg_supressed,1,0)


    #To remove occlusions in ventricle segmentation by choroid plexus (notice the diagonal kernels used)
    open_coronal_brain_mask = cv.morphologyEx(np.uint8(cv.morphologyEx(np.uint8(coronal_brain_mask), 
                                                        cv.MORPH_OPEN,
                                                        np.array([[0,0,1],[0,1,0],[1,0,0]],'uint8'), 
                                                        iterations = 1)),
                                              cv.MORPH_OPEN,
                                              np.array([[1,0,0],[0,1,0],[0,0,1]],'uint8'),
                                              iterations = 1)

    #image y index of the PC's location
    pt = int(coronal_plane.shape[0]-cor_pc_index[2])        

    #produce an initial segmentation by combining the adaptive thresholded and global thresholded images
    ventricles = np.logical_or(np.where(full_brain_mask - open_coronal_brain_mask>0,1,0), 
                               np.where((1-coronal_bin)*full_brain_mask>0,1,0))
    above_pc_mask = np.zeros(ventricles.shape)
    above_pc_mask[0:pt-above_pc_mask_threshold,:] = 1
    ventricles = np.where(ventricles*above_pc_mask>0,1,0) #retain segmentation in an ROI defined by the PC at the inferior level
    
    #this step cleans up non-ventricular tissue by connecting the left LV and right LV if they are disconnected, and then removing extra
    #CSF spaces/sulci/midline that are picked up
    M = cv.moments(np.uint8(full_brain_mask))
    bcX = int(M["m10"] / M["m00"])
    bcY = int(M["m01"] / M["m00"])
    
    middle_roi = np.zeros(ventricles.shape)
    middle_roi[bcY-middle_roi_upper_ylim:bcY, bcX-middle_roi_xlims:bcX+middle_roi_xlims] = 1
    clean_ven_1 = np.logical_or(middle_roi, ventricles)        
    blobs_labels = measure.label(clean_ven_1, background = 0)
    largest_cc_ven = getLargestCC(blobs_labels)      
    [row_v, col_v] = (1-largest_cc_ven).nonzero()
    ventricles_final = ventricles.copy()
    ventricles_final[row_v, col_v] = 0
    
    [vr,vc] = ventricles_final.nonzero()
    lv_roi = np.zeros(ventricles_final.shape)
    iqr_row = np.percentile(vr,75) - np.percentile(vr,25)
    upper_thresh_row = int(np.round((np.percentile(vr,75) + .5*iqr_row)))
    lv_roi[:upper_thresh_row,:] = 1

    
    if (np.sum(ventricles_final)/np.sum(full_brain_mask)) > initial_smooth_ven_fullbrain_ratio:
        ventricles_final_smooth_0 = cv.medianBlur(np.uint8(ventricles_final), median_blur_kernel_size)
    else: ventricles_final_smooth_0 = ventricles_final
    
    dec_thresh = 0
    
    while True: 
        lat_ventricles_smoothed = cv.medianBlur(np.uint8(lv_roi*ventricles_final_smooth_0),smoothing_factor-dec_thresh)
        if np.sum(lat_ventricles_smoothed)/np.sum(full_brain_mask) >= smoothed_ven_fullbrain_ratio: 
            break
        else:
            dec_thresh = dec_thresh + 2 #reducing odd smoothing factors by 2 
            if dec_thresh > dec_smooth_factor_max_thresh:
                print('Ventricles too small, not smoothed')
                lat_ventricles_smoothed = lv_roi*ventricles_final_smooth_0
                break        

    if visualize_intermediate:
        plt.imshow(coronal_plane,cmap = 'gray')
        plt.title('Coronal Plane')
        plt.scatter(np.round(cor_pc_index[0]),cor_img_brain_dat.shape[0]-np.round(cor_pc_index[2]),marker = 'x')
        plt.show()
        plt.imshow(coronal_gauss_blurred,cmap = 'gray')
        plt.title('Coronal Plane - Gauss blurred')
        plt.show()  
        plt.imshow(coronal_bin,cmap = 'gray')
        plt.title('Otsu thresholding')
        plt.show()
        plt.imshow(coronal_adapt_thresh,cmap = 'gray')
        plt.title('Adaptive thresholding')
        plt.show()
        plt.imshow(coronal_brain_mask, cmap = 'gray')
        plt.title('Adaptive threshold background suppressed')
        plt.show()              
        plt.imshow(open_coronal_brain_mask, cmap = 'gray')
        plt.title('Open mask - to eliminate choroid plexus')
        plt.show()        
        plt.imshow(full_brain_mask,cmap = 'gray')
        plt.title('Full brain mask - from closing and floodfill')
        plt.show()
        plt.imshow(ventricles, cmap = 'gray')
        plt.title('Ventricles')
        plt.show()        
        plt.imshow(clean_ven_1, cmap = 'gray')
        plt.title('Venticles connected')
        plt.show()        
        plt.imshow(largest_cc_ven, cmap = 'gray')
        plt.title('LV extracted as one CC')
        plt.show()
        plt.imshow(ventricles_final, cmap = 'gray')        
        plt.title('LV-v0')
        plt.show()
        plt.imshow(ventricles_final_smooth_0, cmap = 'gray')        
        plt.title('LV-v0')
        plt.show()
        plt.imshow(lat_ventricles_smoothed, cmap = 'gray')
        plt.title('LV_Smoothed')
        plt.show()


    try:
        disconnected_case, lat_ventricles = num_connected_comps_cv(lat_ventricles_smoothed)
    except ValueError as e:
        print(f"[Warning] num_connected_comps_cv failed initially ({e}). Attempting fallback smoothing...")
        # Attempt fallback salvage step
        try:
            disconnected_case, lat_ventricles = num_connected_comps_cv(np.uint8(lv_roi * ventricles_final))
        except Exception as fallback_error:
            print(f"[Error] Fallback connected component step also failed: {fallback_error}")
            ca_angle = np.nan
            lat_ventricles = None  
    except Exception as e:
        print(f"[Critical Error] Unexpected failure in num_connected_comps_cv: {e}")
        ca_angle = np.nan
        lat_ventricles = None


    if lat_ventricles is not None:
        try:
            ca_angle = trace_ca_contour(lat_ventricles, coronal_plane, disconnected_case)
        except IndexError as e:
            print(f"[Warning] Contour tracing failed due to over-smoothing ({e}). Applying median blur fallback...")
            try:
                smoothed_fallback = cv.medianBlur(np.uint8(lat_ventricles), median_blur_kernel_size)
                ca_angle = trace_ca_contour(smoothed_fallback, coronal_plane, disconnected_case)
            except Exception as fallback_error:
                print(f"[Error] Fallback contour tracing failed: {fallback_error}")
                ca_angle = np.nan
        except Exception as e:
            print(f"[Critical Error] Unexpected failure in Block 2: {e}")
            ca_angle = np.nan
    else:
        print("[Skipped] contour tracing bypassed because ventricles extraction failed in num_connected_comps_cv")


    return ca_angle

##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
coronal_acpc_vol_aligned_name = config['coronal_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df['acc_num'].unique()

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed CA values for each scan
ca_df = pd.DataFrame()

###### Predefined, emperically determined processing parameters corresponding to ROI sizes, positions, and global/local thresholding parameters. 
The following parameters work for a diverse cohort of chronic neurodegenerative conditions spanning Normal Pressure Hydrocephalus, Alzheimer's disease, post-traumatic encephalomalacia, and headache. So they are recommended to be used directly for patient scans without acute pathology like significant midline shifts, bleeds, or mass effects. 

In [ ]:
processing_parameters = {
    # Preprocessing & Thresholding
    'gauss_blur_kernel_size': 3,                  # Kernel size for initial Gaussian blur
    'adaptive_thresh_size': 75,                   # Neighborhood size for adaptive thresholding
    'adaptive_thresh_C': 7.5,                     # Constant subtracting from the mean for adaptive thresholding
    'window_center': 40,                          # Window center for soft tissue (HU)
    'window_width': 80,                           # Window width for soft tissue (HU)
    'img_dim': 3,                                 # Image dimensionality (3D)

    # ROI & Centroid Clamping
    'above_pc_mask_threshold': 10,                # Places the Lateral Ventricle (LV) ROI 10 mm above the Posterior Commissure (PC)
    'middle_roi_xlims': 20,                       # Non-ventricular tissue beyond +/- 20 mm of the centroid is cleaned up
    'middle_roi_upper_ylim': 15,                  # Non-ventricular tissue beyond 15 mm above the centroid is cleaned up

    # Ventricle Smoothing Strategy
    'smoothing_factor': 7,                        # Starting smoothing factor for all ventricles
    'dec_smooth_factor_max_thresh': 4,            # Maximum threshold by which the smoothing factor can be reduced
    'initial_smooth_ven_fullbrain_ratio': 0.008,  # Smooth ventricles that are at least 0.8% of full brain mask (leaves tiny ventricles unsmoothed)
    'smoothed_ven_fullbrain_ratio': 0.02,         # Accepts the smoothing level if the segmented image is at least 2% of the full brain
    'median_blur_kernel_size': 3                  # Kernel size used for median blurs to produce a smooth boundary for the segmented image
}

##### CA pipeline

In [ ]:
for acc_num in accnum_list: 

    try:
        #assumes the PC's coordinates to be stored in the form of a string read in from excel like '[6.24,-4.3,2.3]'
        pc_str = info_df[pc_coordinates_col_name][(info_df[scan_id_col_name]==acc_num)].values[0]
        [x,y,z] = pc_str.split(",")
        pc_x, pc_y, pc_z = [float(x.strip("[")), float(y), float(z.strip("]"))]   
        pc = [pc_x, pc_y, pc_z]
    except IndexError:
        print(f'PC coordinates for accnum {acc_num} not defined')
 
    print(f'Accession number is {acc_num}')

    cor_img_path = os.path.join(data_path, acc_num, coronal_acpc_vol_aligned_name)
   
    ca_angle = process_calculate_ca(cor_img_path, pc, processing_parameters, visualize_intermediate = False)
        
    ca_df = pd.concat([ca_df, pd.DataFrame({scan_id_col_name:["'" + acc_num + "'"], 'CA':[ca_angle]
                                           })],
                         ignore_index = True)


In [ ]:
len(ca_df), sum(ca_df['CA'].isna())

In [ ]:
#Examine histogram of computed values
plt.hist(ca_df['CA'])
plt.xticks([x for x in np.arange(20, 160, 10)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_ca_lim = 40
max_ca_lim = 150

ca_df[(ca_df['CA'] <= min_ca_lim) | (ca_df['CA'] >= max_ca_lim)]

In [ ]:
ca_df.to_csv(os.path.join(output_folder, 'ca.csv'))